# Rigueur méthodologique : validation par squelette chimique et domaine d'applicabilité

Le ROC AUC de 0,98 obtenu en `04_fingerprint_qsar.ipynb` utilise un split train/test **aléatoire**. Problème classique en QSAR : le jeu de données contient beaucoup d'analogues proches (une même série chimique déclinée en variantes), donc un split aléatoire laisse fuiter des quasi-doublons entre train et test — le modèle a de bonnes chances d'avoir déjà "vu" une molécule presque identique à celle qu'il doit prédire. Le score est alors optimiste.

La pratique standard du domaine (utilisée par exemple dans le benchmark MoleculeNet) est le **split par squelette moléculaire (scaffold split)** : on regroupe les molécules par squelette de Bemis-Murcko et on s'assure qu'un squelette entier reste soit en train, soit en test — jamais réparti entre les deux. Ce notebook mesure l'écart entre les deux méthodologies, honnêtement.

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

df = pd.read_csv("../data/raw/erbb2_activities.csv")
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def compute_descriptors(mol):
    return [
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol), Descriptors.NumRotatableBonds(mol), Descriptors.TPSA(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.RingCount(mol), Descriptors.HeavyAtomCount(mol),
        Descriptors.MolMR(mol),
    ]

def scaffold_of(mol):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return Chem.MolToSmiles(mol)

desc_rows, fp_rows, scaffolds, valid_idx = [], [], [], []
for i, smi in enumerate(df["canonical_smiles"]):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    desc_rows.append(compute_descriptors(mol))
    fp_rows.append(np.array(morgan_gen.GetFingerprint(mol)))
    scaffolds.append(scaffold_of(mol))
    valid_idx.append(i)

X = np.hstack([np.array(desc_rows), np.array(fp_rows)])
y = df.loc[valid_idx, "active"].values
scaffolds = np.array(scaffolds)

print(f"{len(y)} molécules, {len(set(scaffolds))} squelettes chimiques uniques")
pd.Series(scaffolds).value_counts().head(5).rename("nb de molécules")

1070 molécules, 499 squelettes chimiques uniques


c1nc(Nc2ccc(Oc3ccn4ncnc4c3)cc2)c2nc(N3CCNCC3)ccc2n1       66
c1nc(Nc2ccc(Oc3ccc4[nH]cnc4c3)cc2)c2nc(N3CCNCC3)ccc2n1    21
O=C(Nc1ccccc1)Nc1ccc(-c2csc3ccncc23)cc1                   14
c1nc(Nc2ccc(Oc3cnc4[nH]cnc4c3)cc2)c2nc(N3CCNCC3)ccc2n1    12
c1ccc(Cn2cc(-c3ccc4[nH]ncc4c3)nn2)cc1                     11
Name: nb de molécules, dtype: int64

## 1. Validation croisée : split aléatoire vs split par squelette

In [2]:
model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=42,
)

scores_random = cross_val_score(
    model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc"
)
scores_scaffold = cross_val_score(
    model, X, y, cv=GroupKFold(5), groups=scaffolds, scoring="roc_auc"
)

print(f"Split aléatoire (5-fold) : {scores_random.mean():.4f} +/- {scores_random.std():.4f}")
print(f"Split par squelette      : {scores_scaffold.mean():.4f} +/- {scores_scaffold.std():.4f}")
print(f"Écart (optimisme du split aléatoire) : {scores_random.mean() - scores_scaffold.mean():.4f}")

Split aléatoire (5-fold) : 0.9803 +/- 0.0073
Split par squelette      : 0.9723 +/- 0.0092
Écart (optimisme du split aléatoire) : 0.0080


## 2. Hold-out sur squelettes totalement inédits

La validation croisée par squelette (ci-dessus) reste "facile" : les squelettes de test peuvent être chimiquement proches (mêmes familles de motifs) de squelettes vus en train. Pour un test plus dur et plus honnête, on construit un seul split 80/20 en remplissant le train avec les squelettes les plus peuplés en premier (mêmes séries chimiques regroupées), ce qui pousse les squelettes les plus rares — donc les plus différents du reste — vers le test.

In [3]:
groups_idx = defaultdict(list)
for i, s in enumerate(scaffolds):
    groups_idx[s].append(i)
group_list = sorted(groups_idx.values(), key=len, reverse=True)

train_idx, test_idx = [], []
for g in group_list:
    if len(train_idx) + len(g) <= 0.8 * len(y):
        train_idx.extend(g)
    else:
        test_idx.extend(g)

print(f"Train : {len(train_idx)} molécules | Test : {len(test_idx)} molécules (squelettes jamais vus en train)")

scaffold_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=42,
)
scaffold_model.fit(X[train_idx], y[train_idx])
proba = scaffold_model.predict_proba(X[test_idx])[:, 1]

print(f"\nROC AUC hold-out squelettes inédits : {roc_auc_score(y[test_idx], proba):.4f}")
print("(pour référence, le hold-out aléatoire du notebook 04 donnait 0.9812)\n")
print(classification_report(y[test_idx], scaffold_model.predict(X[test_idx])))

Train : 856 molécules | Test : 214 molécules (squelettes jamais vus en train)



ROC AUC hold-out squelettes inédits : 0.9119
(pour référence, le hold-out aléatoire du notebook 04 donnait 0.9812)

              precision    recall  f1-score   support

           0       0.85      0.96      0.90       153
           1       0.86      0.59      0.70        61

    accuracy                           0.86       214
   macro avg       0.86      0.78      0.80       214
weighted avg       0.86      0.86      0.85       214



**Lecture honnête** : sur des squelettes chimiques que le modèle n'a jamais vus, le score chute nettement par rapport au hold-out aléatoire — et surtout le **rappel sur la classe active se dégrade** : le modèle rate une partie significative des molécules réellement actives quand elles appartiennent à une famille chimique inédite. C'est l'information la plus utile pour un usage réel : le modèle est fiable pour prioriser des analogues de séries déjà connues, beaucoup moins pour explorer une chimie franchement nouvelle.

## 3. Domaine d'applicabilité des candidats criblés (notebook 03)

Pour chaque candidat du top 15 généré par recombinaison BRICS (`03_virtual_screening_optimization.ipynb`), on calcule sa similarité de Tanimoto (empreintes de Morgan) à la molécule la plus proche du jeu d'entraînement. Une similarité élevée (>0.6) indique un candidat proche de données connues (score fiable) ; une similarité faible (<0.35) indique une extrapolation — cohérent avec la dégradation observée en section 2, un score élevé sur un candidat très éloigné du train doit être pris avec prudence.

In [4]:
shortlist = pd.read_csv("../data/processed/erbb2_candidate_shortlist.csv")
train_fps = [morgan_gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in df["canonical_smiles"]]

def nearest_neighbor_similarity(smiles):
    fp = morgan_gen.GetFingerprint(Chem.MolFromSmiles(smiles))
    return max(DataStructs.BulkTanimotoSimilarity(fp, train_fps))

shortlist["nn_similarity_to_train"] = shortlist["canonical_smiles"].apply(nearest_neighbor_similarity)

def domain_flag(sim):
    if sim >= 0.6:
        return "proche du train (fiable)"
    if sim >= 0.35:
        return "modérément nouveau"
    return "extrapolation (prudence)"

shortlist["applicability_domain"] = shortlist["nn_similarity_to_train"].apply(domain_flag)
shortlist[["canonical_smiles", "activity_probability", "nn_similarity_to_train", "applicability_domain"]]

,canonical_smiles,activity_probability,nn_similarity_to_train,applicability_domain
0,Cn1cnc2cc(-c3cc4ncnc(-c5ccc6c(c5)ncn6C)c4nc3Oc...,0.997455,0.341176,extrapolation (prudence)
1,Cn1cnc2cc(Oc3ncnc4cc(-c5ccn6ncnc6c5)c(-c5ccc6c...,0.997455,0.413793,modérément nouveau
2,Cc1cc(Oc2ccn3ncnc3c2)ccc1-c1ncnc2ccc(-c3ccn4nc...,0.994146,0.400000,modérément nouveau
3,Cc1c(-c2ccn3ncnc3c2)ccc(-c2ncnc3ccc(Oc4ccn5ncn...,0.994146,0.434783,modérément nouveau
4,Cc1cc(-c2ccc3ncnc(Oc4ccn5ncnc5c4)c3n2)ccc1-c1c...,0.994146,0.382979,modérément nouveau
5,Clc1cc(-c2ccc3ncnc(-c4ccn5ncnc5c4)c3n2)ccc1Oc1...,0.994146,0.464286,modérément nouveau
6,Cc1cc(-c2ncnc3ccc(Oc4ccn5ncnc5c4)nc23)ccc1-c1c...,0.994146,0.420455,modérément nouveau
7,Cc1c(Oc2ccn3ncnc3c2)ccc(-c2ccc3ncnc(-c4ccn5ncn...,0.994146,0.483146,modérément nouveau
8,Cc1c(Oc2ccn3ncnc3c2)ccc(-c2ncnc3ccc(-c4ccn5ncn...,0.994146,0.483146,modérément nouveau
9,Cn1cnc2cc(C3CCCN(c4ncnc5ccc(Oc6ccn7ncnc7c6)nc4...,0.993809,0.401961,modérément nouveau


In [5]:
shortlist.to_csv("../data/processed/erbb2_candidate_shortlist_with_domain.csv", index=False)
print(shortlist["applicability_domain"].value_counts())

applicability_domain
modérément nouveau          14
extrapolation (prudence)     1
Name: count, dtype: int64


## Conclusion

- Le split aléatoire (notebooks 02-04) donnait un ROC AUC ~0,98, optimiste : les molécules de test étaient souvent des quasi-doublons de molécules d'entraînement.
- Sur des squelettes chimiques réellement inédits, la performance se dégrade (ROC AUC ~0,91, rappel actif ~0,6) — c'est le chiffre à citer si on parle de la capacité du modèle à généraliser à de la chimie nouvelle, pas le 0,98.
- Les candidats générés par BRICS (notebook 03) restent, sans surprise, dans une zone "modérément nouvelle" par rapport au train (similarité ~0,3-0,5) — leurs scores d'activité doivent être lus comme des pistes à explorer, pas comme des prédictions fiables au même niveau que sur une molécule proche du train.

Le modèle combiné (descripteurs + empreintes) reste le meilleur choix disponible, mais avec cette lecture plus honnête de ce qu'il sait vraiment prédire.